In [0]:
%run ./04-Streaming-Word-Count

In [0]:
class streamWCTestSuit():
    def __init__(self):
        self.catalog_name='cat_spark_streaming'
        self.schema_name='wcnt'
        self.table_name='wordCountStream'
        self.checkpoint_name='/Volumes/cat_spark_streaming/wcnt/locationforcheckpoint/checkpoint_data/wordcount/'
        self.data_path='/Volumes/cat_spark_streaming/landing/datasets'
        self.test_dataset='/Volumes/cat_spark_streaming/wcnt/sample_datasets/test_datasets'

    def cleanTests(self):
        print(f'Starting Cleanup...', end='')
        spark.sql(f'DROP TABLE IF EXISTS {self.catalog_name}.{self.schema_name}.{self.table_name}')
        dbutils.fs.rm(self.checkpoint_name,True)
        dbutils.fs.rm(self.test_dataset,True)
        dbutils.fs.mkdirs(self.checkpoint_name)
        dbutils.fs.mkdirs(self.test_dataset)
        print(f'Complete')

    def ingestData(self,itr):
        print(f"\tStarting Ingestion...", end= '')
        dbutils.fs.cp(f'{self.data_path}/text_data_{itr}.txt',f'{self.test_dataset}') 
        print('Completed')

    def assertResult(self,expected_count):
        from pyspark.sql.functions import substr
        actual_count=spark.sql(f"SELECT sum(wordCount) FROM {self.catalog_name}.{self.schema_name}.{self.table_name} where substr(word,1,1)=='s'").collect()[0][0]
        assert expected_count == actual_count, f"Test Failed! {actual_count}!={expected_count}"

    def runTests(self):
        import time
        sleepTime=30
        self.cleanTests()
        wc=streamWC()
        

        print("\nTesting first iteration of batch word count ...")
        self.ingestData(1)
        wc.wordCount()
        time.sleep(sleepTime)
        self.assertResult(25)
        print(f'First iteration of batch word count completed')

        
        print("\nTesting second iteration of batch word count ...")
        self.ingestData(2)
        #wc.wordCount()
        time.sleep(sleepTime)
        self.assertResult(5)
        print(f'Second iteration of batch word count completed')

        print("\nTesting Third iteration of batch word count ...")
        self.ingestData(3)
        #wc.wordCount()
        time.sleep(sleepTime)
        self.assertResult(5)
        print(f'Third iteration of batch word count completed')


In [0]:
swcTS=streamWCTestSuit()
swcTS.runTests()

Starting Cleanup...Complete

Testing first iteration of batch word count ...
	Starting Ingestion...Completed
	Starting Word Count Stream...
	Fetching raw data...completed
	Saving table...	Saved table at locationcat_spark_streaming.wcnt.wordCountStream
	Starting Word Count Stream...Completed
First iteration of batch word count completed

Testing second iteration of batch word count ...
	Starting Ingestion...Completed


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-8236643497320269>, line 2
      1 swcTS=streamWCTestSuit()
----> 2 swcTS.runTests()

File <command-8537242214853004>, line 48, in streamWCTestSuit.runTests(self)
     46 #wc.wordCount()
     47 time.sleep(sleepTime)
---> 48 self.assertResult(5)
     49 print(f'Second iteration of batch word count completed')
     51 print("\nTesting Third iteration of batch word count ...")

File <command-8537242214853004>, line 27, in streamWCTestSuit.assertResult(self, expected_count)
     25 from pyspark.sql.functions import substr
     26 actual_count=spark.sql(f"SELECT sum(wordCount) FROM {self.catalog_name}.{self.schema_name}.{self.table_name} where substr(word,1,1)=='s'").collect()[0][0]
---> 27 assert expected_count == actual_count, f"Test Failed! {actual_count}!={expected_count}"

AssertionError: Test Failed! 25!=5

In [0]:
%sql
select * from cat_spark_streaming.wcnt.wordCountStream